In [1]:
import pandas as pd
import numpy as np
import json
import pickle
import warnings
warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE

In [2]:
cards  = pd.read_csv('D:/Downloads/Financial_Transactions/cards_data.csv')
users  = pd.read_csv('D:/Downloads/Financial_Transactions/users_data.csv')
txns   = pd.read_csv('D:/Downloads/Financial_Transactions/sampleTransaction.csv')
mcc  = json.load(open('D:/Downloads/Financial_Transactions/mcc_codes.json'))
labels    = json.load(open('D:/Downloads/Financial_Transactions/train_fraud_labels.json'))['target']


print("Transactions:", txns.shape)
print("Cards:", cards.shape)
print("Users:", users.shape)

Transactions: (532237, 12)
Cards: (6146, 13)
Users: (2000, 14)


In [3]:
labels_df = pd.DataFrame(list(labels.items()), columns=["txn_id", "fraud_label"])


[2] Creating target column...


In [4]:

labels_df["is_fraud"] = labels_df["fraud_label"].map({"Yes": 1, "No": 0})

In [5]:
txns["id"] = txns["id"].astype(str)
labels_df["txn_id"] = labels_df["txn_id"].astype(str)

txns = txns.merge(labels_df, left_on="id", right_on="txn_id", how="left")

In [6]:
txns = txns.dropna(subset=["is_fraud"])

txns["is_fraud"] = txns["is_fraud"].astype(int)

In [7]:
print("Fraud distribution:")
print(txns["is_fraud"].value_counts())

Fraud distribution:
is_fraud
0    356055
1       546
Name: count, dtype: int64


In [8]:
txns["date"] = pd.to_datetime(txns["date"])
txns["hour"] = txns["date"].dt.hour
txns["day"]  = txns["date"].dt.dayofweek
txns["month"] = txns["date"].dt.month

In [9]:
txns["is_night"] = ((txns["hour"] >= 22) | (txns["hour"] <= 5)).astype(int)

In [10]:
def clean_amount(x):
    if isinstance(x, str):
        x = x.replace('$', '').replace(',', '').strip()
        if '(' in x and ')' in x:
            x = '-' + x.replace('(', '').replace(')', '')
    return float(x)

In [11]:
txns["amount"] = txns["amount"].apply(clean_amount)

In [12]:
txns["amount_log"] = np.log1p(txns["amount"])
txns["high_amount"] = (txns["amount"] > 500).astype(int)

In [13]:
HIGH_RISK = ['4829','7995','4511','5815']
txns["high_risk_mcc"] = txns["mcc"].astype(str).isin(HIGH_RISK).astype(int)

In [14]:
df = txns.merge(cards, left_on="card_id", right_on="id", how="left")

In [15]:
print(df.columns)

Index(['id_x', 'date', 'client_id_x', 'card_id', 'amount', 'use_chip',
       'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc',
       'errors', 'txn_id', 'fraud_label', 'is_fraud', 'hour', 'day', 'month',
       'is_night', 'amount_log', 'high_amount', 'high_risk_mcc', 'id_y',
       'client_id_y', 'card_brand', 'card_type', 'card_number', 'expires',
       'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date',
       'year_pin_last_changed', 'card_on_dark_web'],
      dtype='object')


In [16]:
df = df.merge(users, left_on="client_id_x", right_on="id", how="left")

In [17]:
le = LabelEncoder()

In [18]:
for col in ["card_type", "card_brand", "gender"]:
    if col in df.columns:
        df[col] = le.fit_transform(df[col].astype(str))

In [19]:
df = df.fillna(0)

In [20]:
def clean_money(x):
    if isinstance(x, str):
        x = x.replace('$', '').replace(',', '').strip()
        if '(' in x and ')' in x:
            x = '-' + x.replace('(', '').replace(')', '')
    return float(x)

# Apply fix
df["credit_limit"] = df["credit_limit"].apply(clean_money)
df["yearly_income"] = df["yearly_income"].apply(clean_money)

In [21]:
FEATURES = [
    "amount", "amount_log", "hour", "day", "month",
    "is_night", "high_amount", "high_risk_mcc",
    "credit_limit", "card_type", "card_brand",
    "current_age", "yearly_income", "credit_score"
]

FEATURES = [f for f in FEATURES if f in df.columns]

X = df[FEATURES]
y = df["is_fraud"]

In [22]:
print("Total Features:", len(FEATURES))

Total Features: 14


In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [24]:
sm = SMOTE(random_state=42)

In [25]:
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

In [26]:
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)

In [27]:
model.fit(X_train_res, y_train_res)

RandomForestClassifier(max_depth=5, n_jobs=-1, random_state=42)

In [28]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

In [29]:
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.79      0.88     71212
           1       0.00      0.53      0.01       109

    accuracy                           0.79     71321
   macro avg       0.50      0.66      0.44     71321
weighted avg       1.00      0.79      0.88     71321



In [30]:
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

ROC-AUC: 0.7096768042907933


In [31]:
pickle.dump(model, open("fraud_model.pkl", "wb"))

In [32]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "Decision Tree": DecisionTreeClassifier(max_depth=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": xgb.XGBClassifier(eval_metric="logloss")
}

In [ ]:
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train_res, y_train_res)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    auc = roc_auc_score(y_test, y_prob)

    print(f"\n{name} Results:")
    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", auc)

    results[name] = auc


Training Logistic Regression...

Logistic Regression Results:
              precision    recall  f1-score   support

           0       1.00      0.73      0.84     71212
           1       0.00      0.53      0.01       109

    accuracy                           0.73     71321
   macro avg       0.50      0.63      0.42     71321
weighted avg       1.00      0.73      0.84     71321

ROC-AUC: 0.672285544081582

Training Decision Tree...

Decision Tree Results:
              precision    recall  f1-score   support

           0       1.00      0.78      0.88     71212
           1       0.00      0.59      0.01       109

    accuracy                           0.78     71321
   macro avg       0.50      0.68      0.44     71321
weighted avg       1.00      0.78      0.87     71321

ROC-AUC: 0.7204844998291702

Training Random Forest...


In [ ]:
best_model_name = max(results, key=results.get)
print("\nBEST MODEL:", best_model_name)

best_model = models[best_model_name]

In [ ]:
pickle.dump(best_model, open("fraud_model.pkl", "wb"))